# Joe — Google Colab API

Este notebook transforma uma sessão do Google Colab em um backend remoto temporário para o aplicativo desktop Joe.

## O que ele inicia
- **Texto → Áudio:** Kyutai Pocket TTS (CPU).
- **Áudio → Texto:** Moondream Parakeet Redux via Photon (usa CUDA quando uma GPU está disponível).
- **API:** FastAPI protegida por token Bearer.
- **Acesso externo:** Cloudflare Quick Tunnel com URL temporária `trycloudflare.com`.

## Antes de executar
1. Em **Ambiente de execução → Alterar o tipo de ambiente de execução**, escolha GPU quando disponível (por exemplo, T4).
2. Execute as células em sequência.
3. Na última célula, copie a URL da API e o token para a aba **Google Colab / API** do Joe.
4. Não compartilhe a saída da célula com o token. Qualquer pessoa que tenha a URL e o token poderá consumir sua sessão.

**Importante:** o Colab não oferece disponibilidade ou duração garantidas. A sessão e o túnel podem ser encerrados; ao reconectar, gere uma URL e um token novos. <https://research.google.com/colaboratory/faq.html>


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/joe')
REPO_URL = 'https://github.com/g4b3r-desktop/joe.git'
BRANCH = 'feature/colab-model-api'

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    # Atualiza o clone quando o notebook é reutilizado na mesma sessão do Colab.
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

print('Código do Joe pronto:', REPO_DIR)


In [ ]:
%pip install -q -r /content/joe/colab/requirements.txt

import torch
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Aviso: sem GPU disponível; o Parakeet será executado na CPU.')


## Google Drive: carregar modelos e escolher pastas

O Pocket TTS deve usar os arquivos locais extraídos de `Meu Drive/3d_models_ia/JOE_MODELS.zip`. A API não inicia em modo Drive se os pesos e o tokenizer não forem encontrados. O ZIP é extraído para `/content/joe/models/` e reutilizado durante a mesma sessão Colab.

Estrutura esperada (pode estar diretamente na raiz ou dentro de `JOE_MODELS/`):

```text
pocket-tts/
└── languages/portuguese/model.safetensors
pocket-tts-without-voice-cloning/
└── languages/portuguese/
    ├── model.safetensors        # opcional, fallback sem clonagem
    └── tokenizer.json
```

Para síntese com clonagem, o arquivo `pocket-tts/languages/portuguese/model.safetensors` precisa ser o checkpoint **com suporte a voice cloning**. Se o Joe não enviar um áudio de referência na solicitação, informe um WAV local no campo **Voz padrão WAV**. Não é usado um WAV remoto `hf://` como fallback.


In [ ]:
import os
import zipfile
from pathlib import Path

import ipywidgets as widgets
import yaml
from google.colab import drive
from IPython.display import display, clear_output

drive.mount('/content/drive')

DEFAULT_ZIP_PATH = Path('/content/drive/MyDrive/3d_models_ia/modelos-voz/JOE_MODELS.zip')
EXTRACT_BASE = Path('/content/joe/models')

zip_widget = widgets.Text(
    value=str(DEFAULT_ZIP_PATH),
    description='Arquivo ZIP:',
    layout=widgets.Layout(width='95%'),
)
model_folder_widget = widgets.Text(
    value=str(EXTRACT_BASE / 'JOE_MODELS' / 'pocket-tts'),
    description='Pasta Pocket TTS:',
    layout=widgets.Layout(width='95%'),
)
tokenizer_widget = widgets.Text(
    value=str(EXTRACT_BASE / 'JOE_MODELS' / 'pocket-tts-without-voice-cloning' / 'languages' / 'portuguese' / 'tokenizer.json'),
    description='Tokenizer JSON:',
    layout=widgets.Layout(width='95%'),
)
voice_widget = widgets.Text(
    value='',
    placeholder='Obrigatório se o Joe não enviar um WAV: caminho de um WAV no Drive',
    description='Voz padrão WAV:',
    layout=widgets.Layout(width='95%'),
)
hf_cache_widget = widgets.Text(
    value=str(Path('/content/drive/MyDrive/3d_models_ia/huggingface-cache')),
    description='Cache ASR:',
    layout=widgets.Layout(width='95%'),
)
use_drive_tts_widget = widgets.Checkbox(
    value=True,
    description='Obrigatório: carregar Pocket TTS do Google Drive',
)
apply_paths_button = widgets.Button(
    description='Aplicar pastas e modelos',
    button_style='success',
    icon='check',
)
paths_output = widgets.Output()


def _extract_models_zip(zip_path: Path) -> Path:
    if not zip_path.is_file():
        raise FileNotFoundError(f'Arquivo ZIP não encontrado: {zip_path}')

    # Reutiliza extração válida na mesma sessão do Colab.
    for candidate in (EXTRACT_BASE / 'JOE_MODELS', EXTRACT_BASE):
        if (
            (candidate / 'pocket-tts' / 'languages' / 'portuguese' / 'model.safetensors').is_file()
            and any(p.name == 'tokenizer.json' for p in candidate.rglob('tokenizer.json'))
        ):
            return candidate

    EXTRACT_BASE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        base_resolved = EXTRACT_BASE.resolve()
        for member in archive.infolist():
            target = (EXTRACT_BASE / member.filename).resolve()
            if not target.is_relative_to(base_resolved):
                raise ValueError(f'Caminho inseguro dentro do ZIP: {member.filename}')
        archive.extractall(EXTRACT_BASE)

    for candidate in (EXTRACT_BASE / 'JOE_MODELS', EXTRACT_BASE):
        if (candidate / 'pocket-tts').is_dir():
            return candidate
    raise FileNotFoundError(
        'Não encontrei pocket-tts/ dentro do ZIP. Confira a estrutura do JOE_MODELS.zip.'
    )


def apply_drive_model_paths(_=None):
    with paths_output:
        clear_output()
        for key in ('JOE_TTS_CONFIG_PATH', 'JOE_TTS_DEFAULT_VOICE', 'JOE_TTS_USE_DRIVE_MODELS'):
            os.environ.pop(key, None)

        try:
            if not use_drive_tts_widget.value:
                raise RuntimeError(
                    'Este notebook foi configurado para usar os modelos do Drive. '
                    'Marque a opção de carregamento pelo Drive.'
                )

            zip_path = Path(zip_widget.value.strip()).expanduser()
            extracted_root = _extract_models_zip(zip_path)
            model_root = Path(model_folder_widget.value.strip()).expanduser()
            if not model_root.is_dir() or model_root == EXTRACT_BASE / 'JOE_MODELS' / 'pocket-tts':
                model_root = extracted_root / 'pocket-tts'
                model_folder_widget.value = str(model_root)

            clone_weights = model_root / 'languages' / 'portuguese' / 'model.safetensors'
            no_clone_weights = (
                extracted_root / 'pocket-tts-without-voice-cloning'
                / 'languages' / 'portuguese' / 'model.safetensors'
            )
            # O ZIP pode conter pastas extras ou nomes de diretório diferentes.
            # Primeiro respeita o caminho informado; depois procura na árvore extraída.
            tokenizer_file = Path(tokenizer_widget.value.strip()).expanduser()
            if not tokenizer_file.is_file():
                candidates = list(extracted_root.rglob('tokenizer.json'))
                portuguese_candidates = [
                    p for p in candidates if 'portuguese' in str(p).lower()
                ]
                preferred_candidates = [
                    p for p in portuguese_candidates
                    if 'without-voice-cloning' in str(p).lower()
                ]
                search_candidates = preferred_candidates or portuguese_candidates or candidates
                if len(search_candidates) == 1:
                    tokenizer_file = search_candidates[0]
                elif len(search_candidates) > 1:
                    raise FileNotFoundError(
                        'Encontrei vários tokenizer.json. Informe o caminho correto no campo Tokenizer JSON:\\n'
                        + '\\n'.join(str(p) for p in search_candidates)
                    )
                else:
                    raise FileNotFoundError(
                        'Não encontrei tokenizer.json dentro do ZIP extraído. '
                        'Confira se o arquivo foi incluído no JOE_MODELS.zip.'
                    )
                tokenizer_widget.value = str(tokenizer_file)

            missing = [str(p) for p in (clone_weights, tokenizer_file) if not p.is_file()]
            if missing:
                raise FileNotFoundError(
                    'Faltam arquivos obrigatórios no ZIP do Drive:\\n' + '\\n'.join(missing)
                )

            import pocket_tts
            package_config = Path(pocket_tts.__file__).resolve().parent / 'config' / 'portuguese.yaml'
            if not package_config.is_file():
                raise FileNotFoundError(f'Configuração base não encontrada: {package_config}')

            config_data = yaml.safe_load(package_config.read_text(encoding='utf-8'))
            config_data['weights_path'] = str(clone_weights.resolve())
            config_data['weights_path_without_voice_cloning'] = (
                str(no_clone_weights.resolve()) if no_clone_weights.is_file() else None
            )
            config_data['flow_lm']['lookup_table']['tokenizer_path'] = str(tokenizer_file.resolve())

            voice_value = voice_widget.value.strip()
            if voice_value:
                voice_file = Path(voice_value).expanduser()
                if not voice_file.is_file():
                    raise FileNotFoundError(f'WAV de voz padrão não encontrado: {voice_file}')
                if voice_file.suffix.lower() != '.wav':
                    raise ValueError('A voz padrão deve ser um arquivo WAV de referência.')
            else:
                voice_file = None

            runtime_dir = Path('/content/joe/runtime')
            runtime_dir.mkdir(parents=True, exist_ok=True)
            local_config = runtime_dir / 'portuguese_from_drive.yaml'
            local_config.write_text(
                yaml.safe_dump(config_data, sort_keys=False, allow_unicode=True),
                encoding='utf-8',
            )

            # A API só inicia em modo Drive depois que a configuração foi validada.
            os.environ['JOE_TTS_CONFIG_PATH'] = str(local_config)
            os.environ['JOE_TTS_USE_DRIVE_MODELS'] = '1'
            if voice_file:
                os.environ['JOE_TTS_DEFAULT_VOICE'] = str(voice_file.resolve())

            cache_dir = Path(hf_cache_widget.value.strip()).expanduser()
            cache_dir.mkdir(parents=True, exist_ok=True)
            os.environ['HF_HOME'] = str(cache_dir)

            print(f'ZIP extraído em: {extracted_root}')
            print(f'Pesos com clonagem (Drive): {clone_weights}')
            print(
                'Pesos sem clonagem (Drive): '
                + (str(no_clone_weights) if no_clone_weights.is_file() else 'não incluídos')
            )
            print(f'Tokenizer (Drive): {tokenizer_file}')
            print(f'WAV padrão: {voice_file if voice_file else "não definido; o Joe deverá enviar um WAV"}')
            print(f'Configuração local: {local_config}')
            print('Configuração validada: a API usará os pesos locais do Drive.')
        except Exception as exc:
            print(f'ERRO ao configurar os modelos do Drive: {exc}')
            print('A API não deve ser iniciada até corrigir os caminhos/arquivos do ZIP.')


apply_paths_button.on_click(apply_drive_model_paths)
display(widgets.VBox([
    use_drive_tts_widget,
    zip_widget,
    model_folder_widget,
    tokenizer_widget,
    voice_widget,
    hf_cache_widget,
    apply_paths_button,
    paths_output,
]))
apply_drive_model_paths()


## Iniciar a API e o túnel

A API só aceita chamadas com o token gerado nesta sessão. O token não é salvo no repositório nem no disco pelo notebook. Copie-o para o Joe e evite compartilhar esta saída.


In [ ]:
import os
import re
import secrets
import shutil
import subprocess
import sys
import threading
import time
import queue
import requests
from pathlib import Path
from IPython.display import display, Markdown

# Impede iniciar a API se a configuração local do Drive não foi aplicada.
if os.environ.get('JOE_TTS_USE_DRIVE_MODELS') != '1':
    raise RuntimeError(
        'Modelos do Pocket TTS do Google Drive não foram configurados. '
        'Volte à célula Google Drive, corrija os arquivos e clique em Aplicar.'
    )
if not os.environ.get('JOE_TTS_CONFIG_PATH') or not Path(os.environ['JOE_TTS_CONFIG_PATH']).is_file():
    raise RuntimeError('Arquivo YAML local do Pocket TTS não encontrado. Reaplique a configuração do Drive.')

API_TOKEN = secrets.token_urlsafe(36)
os.environ['JOE_API_TOKEN'] = API_TOKEN

# Instala o binário oficial cloudflared para criar um Quick Tunnel temporário.
cloudflared_path = shutil.which('cloudflared')
if cloudflared_path is None:
    binary = '/usr/local/bin/cloudflared'
    subprocess.run(
        ['wget', '-q', '-O', binary,
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
        check=True,
    )
    os.chmod(binary, 0o755)
    cloudflared_path = binary

# Libera a porta 8000 antes de iniciar uma nova instância da API.
result = subprocess.run(
    ["bash", "-c", "fuser -k 8000/tcp"],
    capture_output=True,
    text=True
)

time.sleep(2)

print("Porta 8000 liberada.")

server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api_server:app',
     '--host', '127.0.0.1', '--port', '8000', '--no-access-log'],
    cwd='/content/joe/colab',
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Confirma que a API iniciou antes de publicar o túnel.
api_ready = False
for _ in range(60):
    if server_process.poll() is not None:
        raise RuntimeError('O servidor Joe API terminou inesperadamente.')
    try:
        response = requests.get(
            'http://127.0.0.1:8000/health',
            headers={'Authorization': f'Bearer {API_TOKEN}'},
            timeout=2,
        )
        if response.status_code == 200:
            api_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)
if not api_ready:
    raise RuntimeError('A API não respondeu. Confira a saída do processo uvicorn.')

tunnel_lines = queue.Queue()
tunnel_process = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def _read_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.put(line.rstrip())

threading.Thread(target=_read_tunnel_output, daemon=True).start()
API_URL = None
deadline = time.time() + 90
while time.time() < deadline and API_URL is None:
    if tunnel_process.poll() is not None:
        raise RuntimeError('O cloudflared terminou inesperadamente.')
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        API_URL = match.group(0)

if not API_URL:
    raise RuntimeError('Não foi possível obter a URL trycloudflare. Veja os logs do cloudflared.')

display(Markdown(
    '## Joe API pronta\n\n'
    f'**URL da API:** `{API_URL}`\n\n'
    f'**Token Bearer:** `{API_TOKEN}`\n\n'
    '**Copie os dois valores para a aba Google Colab / API do Joe.** '
    'Não compartilhe esta saída nem publique o token.'
))


## Encerrar a sessão

Execute esta célula quando terminar. Ela encerra o servidor e o túnel; ao iniciar novamente, será gerado outro token e outra URL.


In [ ]:
for process_name in ('tunnel_process', 'server_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
print('Joe API e túnel encerrados.')
